# BERTopic with BoW and Evaluation Metrics
* BoW for vectorization (downweigh overly common words)
* BERT for topic modeling
* Evaluate model quality across different k using coherence score, and topic diversity <br>
--> Plot metrics
* Visualize topic space using PCA & UMAP <br>
--> Present topic clusters and top words for optimal k

In [ ]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import umap

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [2]:
# Load preprocessed data
df = pd.read_pickle("data/preprocessed_data.pkl")
print(f"Loaded {len(df)} posts.")
df.head(5)

Loaded 590938 posts.


,id,author,question,link_flair_css_class,link_flair_text,description,created_utc,year,retrieved_on,org_post,language,clean_question,clean_desc,question_tokens,desc_tokens,q_tuple,d_tuple,post_tokens
0,421zyl,[deleted],How is time defined at the big bang?,physics,Physics,[removed],2016-01-21 20:58:09,2016,2016-02-09 12:21:05,How is time defined at the big bang? [removed],en,how is time defined at the big bang,removed,"[time, define, big, bang]",[],"(time, define, big, bang)",(),"[time, define, big, bang]"
1,4r857d,[deleted],UFO best scientific explanation. Atmospherical...,geo,Earth Sciences,[removed],2016-07-04 17:05:51,2016,2016-09-03 18:04:14,UFO best scientific explanation. Atmospherical...,en,ufo best scientific explanation atmospherical ...,removed,"[ufo, good, scientific, explanation, atmospher...",[],"(ufo, good, scientific, explanation, atmospher...",(),"[ufo, good, scientific, explanation, atmospher..."
2,4r0zwj,De_La_Soul_,It's a commonly known fact that metals oxidize...,chem,Chemistry,,2016-07-03 06:40:06,2016,2016-09-02 17:31:58,It's a commonly known fact that metals oxidize...,en,it s a commonly known fact that metals oxidize...,,"[commonly, know, fact, metal, oxidize, possibl...",[],"(commonly, know, fact, metal, oxidize, possibl...",(),"[commonly, know, fact, metal, oxidize, possibl..."
3,540ohz,MacShakeIt,"China builds large telescope, but i have a que...",astro,Astronomy,[removed],2016-09-22 19:17:55,2016,2016-10-14 11:02:04,"China builds large telescope, but i have a que...",en,china builds large telescope but i have a ques...,removed,"[china, build, large, telescope, question]",[],"(china, build, large, telescope, question)",(),"[china, build, large, telescope, question]"
4,4r191k,[deleted],Geoscientists: Is there any current research i...,geo,Planetary Sci.,[removed],2016-07-03 08:26:38,2016,2016-09-02 17:34:09,Geoscientists: Is there any current research i...,en,geoscientists is there any current research in...,removed,"[geoscientist, current, research, project, sho...",[],"(geoscientist, current, research, project, sho...",(),"[geoscientist, current, research, project, sho..."


In [3]:
df["clean_text"] = df["post_tokens"].apply(lambda x: " ".join(x))
df.head(5)

,id,author,question,link_flair_css_class,link_flair_text,description,created_utc,year,retrieved_on,org_post,language,clean_question,clean_desc,question_tokens,desc_tokens,q_tuple,d_tuple,post_tokens,clean_text
0,421zyl,[deleted],How is time defined at the big bang?,physics,Physics,[removed],2016-01-21 20:58:09,2016,2016-02-09 12:21:05,How is time defined at the big bang? [removed],en,how is time defined at the big bang,removed,"[time, define, big, bang]",[],"(time, define, big, bang)",(),"[time, define, big, bang]",time define big bang
1,4r857d,[deleted],UFO best scientific explanation. Atmospherical...,geo,Earth Sciences,[removed],2016-07-04 17:05:51,2016,2016-09-03 18:04:14,UFO best scientific explanation. Atmospherical...,en,ufo best scientific explanation atmospherical ...,removed,"[ufo, good, scientific, explanation, atmospher...",[],"(ufo, good, scientific, explanation, atmospher...",(),"[ufo, good, scientific, explanation, atmospher...",ufo good scientific explanation atmospherical ...
2,4r0zwj,De_La_Soul_,It's a commonly known fact that metals oxidize...,chem,Chemistry,,2016-07-03 06:40:06,2016,2016-09-02 17:31:58,It's a commonly known fact that metals oxidize...,en,it s a commonly known fact that metals oxidize...,,"[commonly, know, fact, metal, oxidize, possibl...",[],"(commonly, know, fact, metal, oxidize, possibl...",(),"[commonly, know, fact, metal, oxidize, possibl...",commonly know fact metal oxidize possible dize...
3,540ohz,MacShakeIt,"China builds large telescope, but i have a que...",astro,Astronomy,[removed],2016-09-22 19:17:55,2016,2016-10-14 11:02:04,"China builds large telescope, but i have a que...",en,china builds large telescope but i have a ques...,removed,"[china, build, large, telescope, question]",[],"(china, build, large, telescope, question)",(),"[china, build, large, telescope, question]",china build large telescope question
4,4r191k,[deleted],Geoscientists: Is there any current research i...,geo,Planetary Sci.,[removed],2016-07-03 08:26:38,2016,2016-09-02 17:34:09,Geoscientists: Is there any current research i...,en,geoscientists is there any current research in...,removed,"[geoscientist, current, research, project, sho...",[],"(geoscientist, current, research, project, sho...",(),"[geoscientist, current, research, project, sho...",geoscientist current research project shortfal...


In [4]:
documents = df['clean_text'].tolist()
tokenized_docs = df['post_tokens'].tolist()

In [ ]:
"""
#  Define helper functions
def get_top_words_bertopic(model, n_top=15):
    topics = []
    for topic_id in range(len(model.get_topics())):
        if topic_id == -1:
            continue
        terms = model.get_topic(topic_id)
        words = [term[0] for term in terms[:n_top]]
        topics.append(words)
    return topics
"""

In [ ]:
# Define helper functions
def topic_diversity(topics, topk=10):
    top_words = [word for topic in topics for word in topic[:topk]]
    return len(set(top_words)) / len(top_words)

def compute_coherence(topics, texts, dictionary):
    try:
        cm = CoherenceModel(topics=topics, texts=texts, dictionary=dictionary, coherence='c_v')
        return cm.get_coherence()
    except:
        return 0.0

In [ ]:
"""

# Parameter selection
min_topic_sizes = range(5, 71, 5)
embedding_model = "all-MiniLM-L6-v2"
coherences, diversities, n_topics_list = [], [], []
models = {}

dictionary = Dictionary(tokenized_docs)
print(f"Testing {len(min_topic_sizes)} parameter configurations...\n")

for min_size in min_topic_sizes:
    print(f"Fitting BERTopic with min_topic_size={min_size}")
    
    vectorizer_model = CountVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.8, max_features=10000)
    
    model = BERTopic(
        embedding_model=embedding_model,
        min_topic_size=min_size,
        vectorizer_model=vectorizer_model,
        top_n_words=20,
        verbose=False
    )
    
    topics, probs = model.fit_transform(documents)
    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    n_topics_list.append(n_topics)
    
    top_words = get_top_words_bertopic(model, n_top=15)
    coherence = compute_coherence(top_words, tokenized_docs, dictionary)
    diversity = topic_diversity(top_words)
    
    coherences.append(coherence)
    diversities.append(diversity)
    models[min_size] = model
    
    print(f"  → Topics: {n_topics} | Coherence: {coherence:.4f} | Diversity: {diversity:.4f}")

"""

Testing 14 parameter configurations...

Fitting BERTopic with min_topic_size=5


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
from sentence_transformers import SentenceTransformer

# Parameter selection
min_topic_sizes = range(5, 71, 5)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Computing embeddings...")
embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True,
    batch_size=64
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Computing embeddings...


Batches:   0%|          | 0/9234 [00:00<?, ?it/s]

Testing 14 parameter configurations...

Fitting BERTopic with min_topic_size=5


TypeError: 'bool' object is not subscriptable

In [ ]:
coherences, diversities, n_topics_list = [], [], []
models = {}

dictionary = Dictionary(tokenized_docs)
print(f"Testing {len(min_topic_sizes)} parameter configurations...\n")

for min_size in min_topic_sizes:
    print(f"Fitting BERTopic with min_topic_size={min_size}")
    
    vectorizer_model = CountVectorizer(
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.8,
        max_features=10000
    )
    
    model = BERTopic(
        min_topic_size=min_size,
        vectorizer_model=vectorizer_model,
        top_n_words=20,
        verbose=False
    )
    
    topics, probs = model.fit_transform(
        documents,
        embeddings=embeddings
    )
    
    # number of topics (excluding outliers)
    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    n_topics_list.append(n_topics)
    
    top_words = [
        [word for word, _ in words[:15]]
        for topic, words in model.get_topics().items()
        if topic != -1 and words
    ]
    
    # metrics
    if top_words:
        coherence = compute_coherence(top_words, tokenized_docs, dictionary)
        diversity = topic_diversity(top_words)
    else:
        coherence = 0
        diversity = 0
    
    coherences.append(coherence)
    diversities.append(diversity)
    models[min_size] = model
    
    print(f"  → Topics: {n_topics} | Coherence: {coherence:.4f} | Diversity: {diversity:.4f}")

Testing 14 parameter configurations...

Fitting BERTopic with min_topic_size=5
  → Topics: 11902 | Coherence: 0.0000 | Diversity: 0.0840
Fitting BERTopic with min_topic_size=10


In [ ]:
# Plot results & Select best model
results_df = pd.DataFrame({
    'min_topic_size': min_topic_sizes,
    'n_topics': n_topics_list,
    'coherence': coherences,
    'diversity': diversities
})

scaler = MinMaxScaler()
results_df['coherence_norm'] = scaler.fit_transform(results_df[['coherence']])
results_df['diversity_norm'] = scaler.fit_transform(results_df[['diversity']])
results_df['combined_score'] = 0.6 * results_df['coherence_norm'] + 0.4 * results_df['diversity_norm']

print(results_df.to_string(index=False))

best_idx = results_df['combined_score'].idxmax()
best_min_size = results_df.loc[best_idx, 'min_topic_size']
best_coherence = results_df.loc[best_idx, 'coherence']
best_diversity = results_df.loc[best_idx, 'diversity']
best_n_topics = results_df.loc[best_idx, 'n_topics']
best_score = results_df.loc[best_idx, 'combined_score']

print(f"\n{'='*60}\nBEST CONFIGURATION:\n  min_topic_size: {int(best_min_size)}\n  n_topics: {int(best_n_topics)}\n  Coherence: {best_coherence:.4f}\n  Diversity: {best_diversity:.4f}\n  Combined Score: {best_score:.4f}\n{'='*60}")

In [ ]:
# Plot metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('BERTopic Parameter Selection: Evaluation Metrics', fontsize=16, fontweight='bold')

axes[0, 0].plot(results_df['min_topic_size'], results_df['coherence'], marker='o', linewidth=2, markersize=8, color='#2E86AB')
axes[0, 0].scatter([best_min_size], [best_coherence], s=200, color='red', zorder=5, label='Best')
axes[0, 0].set_xlabel('min_topic_size')
axes[0, 0].set_ylabel('Coherence')
axes[0, 0].set_title('Coherence Score (higher is better)')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend()

axes[0, 1].plot(results_df['min_topic_size'], results_df['diversity'], marker='s', linewidth=2, markersize=8, color='#A23B72')
axes[0, 1].scatter([best_min_size], [best_diversity], s=200, color='red', zorder=5, label='Best')
axes[0, 1].set_xlabel('min_topic_size')
axes[0, 1].set_ylabel('Diversity')
axes[0, 1].set_title('Topic Diversity (higher is better)')
axes[0, 1].grid(alpha=0.3)
axes[0, 1].legend()

axes[1, 0].plot(results_df['min_topic_size'], results_df['n_topics'], marker='^', linewidth=2, markersize=8, color='#F18F01')
axes[1, 0].scatter([best_min_size], [best_n_topics], s=200, color='red', zorder=5, label='Best')
axes[1, 0].set_xlabel('min_topic_size')
axes[1, 0].set_ylabel('Number of Topics')
axes[1, 0].set_title('Number of Discovered Topics')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].legend()

axes[1, 1].plot(results_df['min_topic_size'], results_df['combined_score'], marker='d', linewidth=2, markersize=8, color='#C73E1D')
axes[1, 1].scatter([best_min_size], [best_score], s=200, color='red', zorder=5, label='Best')
axes[1, 1].set_xlabel('min_topic_size')
axes[1, 1].set_ylabel('Combined Score')
axes[1, 1].set_title('Combined Score (0.6*Coherence + 0.4*Diversity)')
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('viz/bertopic_parameter_selection.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Best Model Details
best_model = models[best_min_size]
best_topics, best_probs = best_model.fit_transform(documents)

print(f"Best BERTopic Model Configuration:\n  - min_topic_size: {int(best_min_size)}\n  - Number of topics: {int(best_n_topics)}\n  - Coherence: {best_coherence:.4f}\n  - Diversity: {best_diversity:.4f}\n")
print("Top 5 topics and words:\n" + "="*80)
for topic_id in range(min(5, len(best_model.get_topics()))):
    if topic_id == -1:
        continue
    terms = best_model.get_topic(topic_id)
    top_words = [f"{word} ({weight:.3f})" for word, weight in terms[:10]]
    print(f"\nTopic {topic_id}:\n  {', '.join(top_words)}")

# Cell 9: BERTopic Visualizations
print("Generating BERTopic visualizations...\n")
try:
    fig = best_model.visualize_topics()
    fig.write_html("viz/bertopic_topics.html")
    print("✓ Saved: viz/bertopic_topics.html")
except Exception as e:
    print(f"Could not generate topic visualization: {e}")

try:
    fig = best_model.visualize_barchart()
    fig.write_html("viz/bertopic_barchart.html")
    print("✓ Saved: viz/bertopic_barchart.html")
except Exception as e:
    print(f"Could not generate barchart: {e}")

try:
    fig = best_model.visualize_heatmap()
    fig.write_html("viz/bertopic_heatmap.html")
    print("✓ Saved: viz/bertopic_heatmap.html")
except Exception as e:
    print(f"Could not generate heatmap: {e}")

# Cell 10: UMAP Visualization
embeddings = best_model._extract_embeddings(documents, method="document")
print("Computing UMAP for embeddings...")
umap_model = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
umap_embeddings = umap_model.fit_transform(embeddings)

fig, ax = plt.subplots(figsize=(14, 10))
scatter = ax.scatter(umap_embeddings[:, 0], umap_embeddings[:, 1], c=best_topics, cmap='tab20', alpha=0.6, s=30, edgecolors='k', linewidth=0.2)
ax.set_xlabel('UMAP Dimension 1', fontsize=12)
ax.set_ylabel('UMAP Dimension 2', fontsize=12)
ax.set_title(f'BERTopic Document Embedding Space (k={int(best_n_topics)})', fontsize=14, fontweight='bold')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Topic ID', fontsize=11)
plt.tight_layout()
plt.savefig('viz/bertopic_embedding_space.png', dpi=300, bbox_inches='tight')
plt.show()

# Cell 11: Save Results
best_model.save("topics/bertopic_best_model")
results_df.to_csv("topics/bertopic_parameter_selection_results.csv", index=False)
topic_assignments = pd.DataFrame({
    'doc_id': range(len(documents)),
    'topic': best_topics,
    'confidence': best_probs.max(axis=1) if hasattr(best_probs, 'shape') and len(best_probs.shape) > 1 else best_probs
})
topic_assignments.to_csv("topics/bertopic_topic_assignments.csv", index=False)
print("✓ Saved all results")